# Income Classification using Logistic Regression

Goal: Predict whether a person earns more than $50K a year, based on details like age, education, job and hours worked per week.

Dataset: Adult Census Income (from Hugging Face)

Steps: Load data, clean it, split it, train a model, and evaluate it.

# Step 1: Load the Dataset
I loaded the Adult Census Income dataset from Hugging Face and converted it into a Pandas table (DataFrame). Each row is one person, and each column is a detail about them. The `income` column is what we want to predict.

In [1]:
import numpy as np
import pandas as pd
from datasets import load_dataset

ds = load_dataset("scikit-learn/adult-census-income")
df = ds["train"].to_pandas()
df.head()

C:\Users\ZainabJunaid\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


# Step 2: Clean the Data
Missing values in this dataset are written as `?`, so I replaced them with proper empty values (NaN). Then I removed rows with missing values and removed duplicate rows. A model learns better from clean data, and it cannot handle missing values directly.

In [2]:
df = df.replace("?", np.nan)
print(df.isnull().sum())
df = df.dropna().drop_duplicates()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64


# Step 3: Prepare Features and Target
Target (y): whether the person earns more than $50K. I turned it into numbers: 1 = more than $50K, 0 = $50K or less.
Features (X): all the other columns. Models only understand numbers, so I converted text columns (like occupation) into 0/1 columns using `get_dummies`.

In [3]:
y = (df["income"] == ">50K").astype(int)
X = pd.get_dummies(df.drop("income", axis=1), drop_first=True)

# Step 4: Prepare Features and Target
I split the data into 80% for training and 20% for testing. The model learns from the training part, then I check it on the test part, which it has never seen. This shows how well it will work on new people, like an exam after studying. `random_state=42` makes the split the same every time I run it.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 5: Scale and Train the Model
Scaling: Columns have very different ranges (for example, age is about 20 to 90 but capital gain can be thousands). `StandardScaler` puts them on a similar scale so no column dominates. I fit the scaler on training data only, so no information from the test data leaks in.
Model: I used Logistic Regression, a simple classifier. It calculates the probability that a person earns more than $50K and predicts ">50K" if that probability is above 50%.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

# Step 6: Evaluate the Model

Confusion matrix shows how many predictions were right and wrong:

 Predicted <=50K | Predicted >50K |
 Actually <=50K | 4260 (TN) | 320 (FP) |
 Actually >50K | 564 (FN) | 884 (TP) |

TN (True Negative): predicted <=50K and it was right
FP (False Positive): predicted >50K but the person actually earns <=50K
FN (False Negative): predicted <=50K but the person actually earns >50K
TP (True Positive): predicted >50K and it was right

Accuracy = (TP + TN) / total = (884 + 4260) / 6028 = 0.853
Out of all predictions, 85% were correct.

Precision = TP / (TP + FP) = 884 / (884 + 320) = 0.734
When the model says "earns >50K", it is right 73% of the time.

Recall = TP / (TP + FN) = 884 / (884 + 564) = 0.610
Out of all people who really earn >50K, the model found 61% of them.

In [6]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, confusion_matrix)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8533510285335103
Precision: 0.7342192691029901
Recall: 0.6104972375690608
[[4260  320]
 [ 564  884]]


# Conclusion
The model reached about 85% accuracy. Its precision (73%) is better than its recall (61%), so it is careful when it predicts ">50K", but it misses about 39% of the people who actually earn more than $50K. One reason is that the data has more people earning <=50K than >50K, so the model leans toward the bigger group.

Possible improvements: try other models (like Random Forest), handle the class imbalance, or tune the model settings.